In [ ]:
import os
import cv2   
import numpy as np 
from tqdm import tqdm
from PIL import Image 
from glob import glob  
import tensorflow as tf    

In [4]:
base_path = r"D:\final year project\all datasets\original\Second Set"
preprocessed_path1 = r"D:\final year project\all datasets\original\resized_set"
preprocessed_path2 = r"D:\final year project\all datasets\original\denoised_set"
preprocessed_path3 = r"D:\final year project\all datasets\original\normalized_set"
preprocessed_path4 = r"D:\final year project\all datasets\original\enhanced_set" 
os.makedirs(preprocessed_path1, exist_ok=True)
os.makedirs(preprocessed_path2, exist_ok=True)
os.makedirs(preprocessed_path3, exist_ok=True)
os.makedirs(preprocessed_path4, exist_ok=True)

In [5]:
# RESIZE AUGMENTED IMAGES

TARGET_SIZE = (256, 256)

def resize_dataset(input_dir, output_dir):
    for class_name in ["400x Normal Oral Cavity Histopathological Images", "400x OSCC Histopathological Images"]:
        os.makedirs(os.path.join(output_dir, class_name), exist_ok=True)
        input_path = os.path.join(input_dir, class_name)
        output_path = os.path.join(output_dir, class_name)
        
        img_files = [f for f in os.listdir(input_path) if f.lower().endswith('.jpg')]
        
        for img_file in tqdm(img_files, desc=f"Resizing {class_name}"):
            try:
                img = cv2.imread(os.path.join(input_path, img_file))
                if img is not None:
                    resized = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
                    cv2.imwrite(os.path.join(output_path, img_file), resized)
            except Exception:
                continue

print("Resizing images...")
resize_dataset(base_path, preprocessed_path1)
print(f"\nResizing complete. Output saved to: {preprocessed_path1}")

Resizing images...


Resizing 400x Normal Oral Cavity Histopathological Images: 100%|██████████| 201/201 [00:23<00:00,  8.52it/s]
Resizing 400x OSCC Histopathological Images: 100%|██████████| 495/495 [00:57<00:00,  8.67it/s]


Resizing complete. Output saved to: D:\final year project\all datasets\original\resized_set


In [6]:

# Apply median filter to color image
def median_filter_color(img, kernel_size=3):
    # Ensure kernel size is odd and >= 3
    kernel_size = max(3, kernel_size | 1)
    return cv2.medianBlur(img, kernel_size)

# Denoise function using median filter
def denoise_dataset(input_dir, output_dir, kernel_size=3):
    os.makedirs(output_dir, exist_ok=True)
    
    # Get all class folders
    class_folders = [d for d in os.listdir(input_dir) if os.path.isdir(os.path.join(input_dir, d))]
    
    for class_name in tqdm(class_folders, desc="Processing classes"):
        class_input_dir = os.path.join(input_dir, class_name)
        class_output_dir = os.path.join(output_dir, class_name)
        os.makedirs(class_output_dir, exist_ok=True)
        
        # Get all images in class folder
        image_files = [f for f in os.listdir(class_input_dir) if f.lower().endswith(('.jpg'))]
        
        for img_file in tqdm(image_files, desc=f"Processing {class_name}", leave=False):
            img_path = os.path.join(class_input_dir, img_file)
            
            # Load image
            img = cv2.imread(img_path)
            if img is None:
                print(f"Warning: Could not read image {img_path}")
                continue
            
            # Apply median filter
            filtered_img = median_filter_color(img, kernel_size=kernel_size)
            
            # Save result
            output_path = os.path.join(class_output_dir, img_file)
            cv2.imwrite(output_path, filtered_img)

# Example usage
if __name__ == "__main__":
    denoise_dataset(preprocessed_path1, preprocessed_path2, kernel_size=3)

Processing classes: 100%|██████████| 2/2 [00:37<00:00, 18.70s/it]


In [7]:
#Stain Normalization:

# Optical Density (OD) normalization constants
Io = 240        # Light intensity used in image formation
alpha = 1       # Percentile threshold to exclude extreme angles in stain vectors
beta = 0.15     # Threshold for OD filtering


# Reference H&E stain matrix (predefined based on a reference image)
HERef = tf.constant([[0.5626, 0.2159],
                     [0.7201, 0.8012],
                     [0.4062, 0.5581]], dtype=tf.float32) 

maxCRef = np.array([1.9705, 1.0308])   # Reference stain concentration percentiles


def process_single_image(img):
    h, w, _ = img.shape   # Get image dimensions (height, width, channels)
    img_flat = img.reshape((-1, 3))  # Flatten image to 2D array of pixels (N, 3)

    OD = -tf.math.log((tf.cast(img_flat, tf.float32) + 1) / Io) # Convert RGB to optical density (OD)

    mask = tf.reduce_all(OD >= beta, axis=1) # Mask to filter out low OD values (background or noise)
    
    ODhat = tf.boolean_mask(OD, mask) # Apply mask to retain only meaningful stain regions

    # Perform PCA on ODhat to find principal stain directions
    eigvals, eigvecs = np.linalg.eigh(np.cov(ODhat.numpy().T))
    eigvecs_tf = tf.convert_to_tensor(eigvecs, dtype=tf.float32)

    
    That = tf.matmul(ODhat, eigvecs_tf[:, 1:3]) # Project ODhat onto 2 principal components

    
    phi = tf.math.atan2(That[:, 1], That[:, 0]).numpy() # Compute angles (phi) of projected data for stain vector estimation

    # Compute angle percentiles to exclude outliers
    minPhi = np.percentile(phi, alpha)
    maxPhi = np.percentile(phi, 100 - alpha)

    # Compute stain vectors based on min and max angle projections
    vMin = eigvecs_tf[:, 1:3] @ tf.constant([[np.cos(minPhi)], [np.sin(minPhi)]], dtype=tf.float32)
    vMax = eigvecs_tf[:, 1:3] @ tf.constant([[np.cos(maxPhi)], [np.sin(maxPhi)]], dtype=tf.float32)

    # Order stain vectors: Hematoxylin first, Eosin second
    HE = tf.concat([vMin, vMax], axis=1) if vMin[0] > vMax[0] else tf.concat([vMax, vMin], axis=1)
    
    # Solve OD = HE * C using least squares to get stain concentrations
    Y = tf.transpose(OD)
    C = tf.linalg.lstsq(HE, Y, l2_regularizer=1e-10)

    
    C_np = C.numpy() # Convert concentrations to numpy for further processing

    # Normalize stain concentrations using 99th percentile
    maxC = np.array([np.percentile(C_np[0, :], 99), np.percentile(C_np[1, :], 99)])
    tmp = maxC / maxCRef
    C2 = C_np / tmp[:, np.newaxis]
    C2_tf = tf.convert_to_tensor(C2, dtype=tf.float32)

    # Reconstruct normalized image using reference stain matrix
    Inorm = Io * tf.exp(-tf.matmul(HERef, C2_tf))
    Inorm = tf.clip_by_value(Inorm, 0, 255)

    # Reshape and convert back to image format (uint8)
    Inorm_img = tf.reshape(tf.transpose(Inorm), (h, w, 3)).numpy().astype(np.uint8)
    
    return Inorm_img


def process_directory(input_dir, output_dir):
    class_folders = [d for d in os.listdir(input_dir) if os.path.isdir(os.path.join(input_dir, d))]
    for class_name in class_folders:
        input_class_path = os.path.join(input_dir, class_name)
        output_class_path = os.path.join(output_dir, class_name)
        os.makedirs(output_class_path, exist_ok=True)

        for file in tqdm(os.listdir(input_class_path), desc=f"Processing {class_name}"):
            if file.lower().endswith(('.jpg')):
                in_path = os.path.join(input_class_path, file)
                try:
                    img = Image.open(in_path).convert("RGB") # Load image and convert to RGB numpy array
                    img_np = np.array(img)
                    
                    norm_img = process_single_image(img_np)   # Perform stain normalization

                    # Save normalized image to output directory
                    out_path = os.path.join(output_class_path, os.path.splitext(file)[0] + "_normalized.jpg")
                    Image.fromarray(norm_img).save(out_path)
                    
                except Exception as e:
                    print(f"❌ Failed to process {in_path}: {e}")

process_directory(preprocessed_path2, preprocessed_path3)

Processing 400x Normal Oral Cavity Histopathological Images: 100%|██████████| 201/201 [00:14<00:00, 14.01it/s]
Processing 400x OSCC Histopathological Images: 100%|██████████| 495/495 [00:31<00:00, 15.95it/s]


In [14]:
#CLAHE

# Create output directory structure
class_dirs = [d for d in os.listdir(preprocessed_path3) if os.path.isdir(os.path.join(preprocessed_path3, d))]
for class_dir in class_dirs:
    os.makedirs(os.path.join(preprocessed_path4, class_dir), exist_ok=True)

# Apply CLAHE using OpenCV
def apply_clahe_to_image(image_np):
    lab = cv2.cvtColor(image_np, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.7, tileGridSize=(8,8))
    l_clahe = clahe.apply(l)
    lab_clahe = cv2.merge((l_clahe, a, b))
    img_clahe = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2RGB)
    return img_clahe

# Process all images
for class_dir in class_dirs:
    input_class_path = os.path.join(preprocessed_path3, class_dir)
    output_class_path = os.path.join(preprocessed_path4, class_dir)

    image_paths = glob(os.path.join(input_class_path, "*.jpg"))

    for image_path in tqdm(image_paths, desc=f"Processing {class_dir}"):
 
        image = tf.io.read_file(image_path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.convert_image_dtype(image, tf.uint8)

        image_np = image.numpy()
        
        # Apply CLAHE
        clahe_image = apply_clahe_to_image(image_np)

        # Save output
        image_name = os.path.basename(image_path)
        output_path = os.path.join(output_class_path, image_name)
        cv2.imwrite(output_path, cv2.cvtColor(clahe_image, cv2.COLOR_RGB2BGR)) 

Processing 400x Normal Oral Cavity Histopathological Images: 100%|██████████| 201/201 [00:04<00:00, 46.56it/s]
Processing 400x OSCC Histopathological Images: 100%|██████████| 495/495 [00:07<00:00, 62.69it/s]
